# Test multiple variants and metrics in A/B/n

[All How-to Guides](README.md)

Compare revenue and order status across three groups. Some orders have a new status that appears in only one variant. Read the results under the Holm and Benjamini–Hochberg adjustments.

## 1. Inspect the frequency table

Create one numeric and one categorical observation per user. The `manual_review` status occurs only in variant 2. Keep the category in the control frequency table with a count of zero.

In [1]:
from IPython.display import display

import numpy as np
import pandas as pd

from hypex import ABTest
from hypex.dataset import Dataset, TargetRole, TreatmentRole

rng = np.random.default_rng(101)
status_counts = [
    {"paid": 360, "abandoned": 240},
    {"paid": 364, "abandoned": 236},
    {"paid": 355, "abandoned": 205, "manual_review": 40},
]
frame = pd.concat(
    [
        pd.DataFrame(
            {
                "variant": variant,
                "revenue": rng.normal(40 + lift, 12, 600),
                "status": rng.permutation(
                    np.repeat(list(counts), list(counts.values()))
                ),
            }
        )
        for variant, (lift, counts) in enumerate(zip([0, 1.2, 4], status_counts))
    ],
    ignore_index=True,
)
pd.crosstab(frame["variant"], frame["status"])

status,abandoned,manual_review,paid
variant,,,
0,240,0,360
1,236,0,364
2,205,40,355


`manual_review` appears 40 times in variant 2 and is absent in the other two groups. Before interpreting the chi-square test, check that its expected cell counts are large enough.

## 2. Choose an adjustment for your analysis

`holm` controls the probability of at least one false rejection in a family. `fdr_bh` controls the expected proportion of false discoveries under independence or appropriate positive dependence among the tests. Specify the method and the family in advance. In this report, HypEx adjusts t-test and chi-square p-values separately.

`additional_tests` selects the t-test and chi-square test for the two metric types. `multitest_method` specifies the p-value adjustment. The numeric metric uses Welch's t-test by default.

`holm` is the default `multitest_method`. The example includes it explicitly to compare it with `fdr_bh`.

In each full `multitest` report, find `old p-value`, `new p-value`, and `rejected`.

In [2]:
data = Dataset(
    roles={
        "variant": TreatmentRole(int),
        "revenue": TargetRole(float),
        "status": TargetRole(str),
    },
    data=frame,
)
for method in ("holm", "fdr_bh"):
    result = ABTest(
        additional_tests=["t-test", "chi2-test"],
        multitest_method=method,
    ).execute(data)
    print(method)
    display(result.multitest)

holm


,field,test,old p-value,new p-value,correction,rejected,group
0,revenue,TTest,0.01197821929093111,0.01197821929093111,1.0,True,1
1,revenue,TTest,5.353867110602919e-12,1.0707734221205838e-11,0.5,True,2
2,status,Chi2Test,0.9334927366713109,0.9334927366713109,1.0,False,1
3,status,Chi2Test,2.2613951798738886e-05,4.522790359747777e-05,0.5,True,2


fdr_bh


,field,test,old p-value,new p-value,correction,rejected,group
0,revenue,TTest,0.01197821929093111,0.01197821929093111,1.0,True,1
1,revenue,TTest,5.353867110602919e-12,1.0707734221205838e-11,0.5,True,2
2,status,Chi2Test,0.9334927366713109,0.9334927366713109,1.0,False,1
3,status,Chi2Test,2.2613951798738886e-05,4.522790359747777e-05,0.5,True,2


`group` identifies the variant compared with the shared control. For `status`, the chi-square test compares entire distributions. Return to the frequency table to see which categories changed. Holm and Benjamini–Hochberg give the same decisions in this example. Choose the method based on the analysis objective and the assumptions about dependence among the tests.

To compare variants on one metric, see the [shared-control A/B/n guide](abn-variants.ipynb).

See the [ABTest tutorial](../quick_tutorials/ABTest.ipynb#ab-holm) for the Holm procedure and its assumptions.